# Consulta de ocupación del calendario de Airbnb en Málaga.

En este notebook realizamos una consulta de la ocupación de los alojamientos de Airbnb en Málaga, utilizando una ventana deslizante de 30 días con un desplazamiento de 7 días.

Nuestro objetivo con esta consutla es poder analizar la evolución teporal de la ocupación de los alojamientos para ello, vamos a calcular:

- El porcentaje medio de la ocupación, el cual nos va a indicar la fracción de los alojamientos que estuvieron reservados de media.
- El total de las reservas.

In [ ]:
import sys
import pathlib
notebook_dir = pathlib.Path.cwd()
project_root = notebook_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f'Directorio raíz añadido al path: {project_root}')

In [ ]:
from src.kafka.consumer_kafka import create_kafka_stream_df, consumer_kafka_avro
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, sum as spark_sum

In [ ]:
consumer_kafka_avro(
    topic = 'aribnb_calendar_gold',
    idle_timeout_seconds = 10,
    log_to_file = False
)

In [ ]:
spark = (SparkSession.builder
        .appName("Notebook_Calendar_SlidingWindow")
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark Session iniaca con éxito.")

In [ ]:
df_calendar_raw = create_kafka_stream_df(spark, "airbnb_calendar_gold")

df_processed = (df_calendar_raw
        .withColumn("event_timestamp", col("date").cast("timestamp"))
        .withColumn("booked", col("booked").cast("integer"))
)

df_group = (df_processed
        .withWatermark("event_timestamp", "7 days")
        .groupBy(
            window(col("event_timestamp"), "30 days", "7 days")
        ).agg(
            (avg(col("booked")) * 100).alias("ocupacion_media_pct"),
            spark_sum(col("booked")).alias("reservas_totales")
        )
)

df_final = df_group.orderBy(col("window.start").desc())

In [ ]:
query_calendar = (df_final.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("tabla_resultados_calendar")
        .start()
)

In [ ]:
spark.sql("SELECT * FROM talba_resultados_calendar").show(truncate=False)